In [9]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAI
from dotenv import load_dotenv


In [10]:
load_dotenv()

True

In [11]:
llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

In [12]:
class LLMState(TypedDict):
    question:str
    answer:str

In [13]:
def llm_qa(state:LLMState)->LLMState:
    question=state['question']
    prompt=f'Answer the following question {question}'
    answer=llm.invoke(prompt).text
    state['answer']=answer
    
    return state


In [18]:
graph=StateGraph(LLMState)
graph.add_node('llm_qa',llm_qa)
graph.add_edge(START,"llm_qa")
graph.add_edge('llm_qa',END)
workflow=graph.compile()

In [20]:
initial_state={'question':'What is langgraph ?'}
final_state=workflow.invoke(initial_state)
print(final_state['answer'])

**LangGraph** is an open-source library developed by the creators of LangChain, designed for building **stateful, multi-actor applications with Large Language Models (LLMs)**. 

In simple terms, LangGraph allows you to create complex AI "agents" that can work in loops, make decisions, correct their own mistakes, and collaborate with other agents or humans.

Here is a comprehensive breakdown of what LangGraph is, why it is important, and how it works.

---

### 1. Why do we need LangGraph? (The Problem it Solves)
Traditional LLM frameworks (like standard LangChain or LlamaIndex) are built on **DAGs** (Directed Acyclic Graphs). This means data flows in one direction: 
`Prompt ➡️ LLM ➡️ Tool ➡️ Output`.

However, real-world AI agents require **cycles (loops)** and complex decision-making. For example:
* **Self-Correction:** An LLM writes code, runs a test, sees an error, and loops back to rewrite the code until the test passes.
* **Human-in-the-loop:** An AI drafts an email, pauses to ask